# Day 08 RAG Pipeline Demo: Task 1 -> Task 10

Notebook này là một mẫu chạy từng bước để xem luồng dữ liệu từ raw documents/news -> markdown -> chunks -> retrieval -> generation có citation.

Mặc định các bước tốn API hoặc ghi index được tắt. Bật các flag ở cell Setup nếu muốn chạy đầy đủ.

In [45]:
from pathlib import Path
import json
import os
import sys

# Nếu mở notebook từ thư mục notebooks/, đưa project root vào sys.path.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Flags: đổi True nếu muốn chạy lại bước ghi file/API/index.
RUN_TASK1_DOWNLOAD = True
RUN_TASK2_CRAWL = True
RUN_TASK3_CONVERT = True
RUN_TASK4_INDEX = True       # Gọi OpenAI embedding + ghi Weaviate
RUN_SEMANTIC_SEARCH = True   # Gọi OpenAI embedding query + Weaviate
RUN_PAGEINDEX_UPLOAD = True  # Gọi PageIndex API
RUN_GENERATION = True        # Gọi LLM nếu có OPENAI_API_KEY

print("Project root:", PROJECT_ROOT)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("PAGEINDEX_API_KEY set:", bool(os.getenv("PAGEINDEX_API_KEY")))

Project root: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2
OPENAI_API_KEY set: True
PAGEINDEX_API_KEY set: True


## Task 1 - Thu thập văn bản pháp luật

Mục tiêu: tải hoặc kiểm tra các file PDF/DOCX trong `data/landing/legal/`.

In [46]:
from src.task1_collect_legal_docs import DATA_DIR as LEGAL_DIR, download_all

if RUN_TASK1_DOWNLOAD:
    download_all(overwrite=False)

legal_files = sorted([p for p in LEGAL_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".pdf", ".doc", ".docx"}])
print(f"Legal files: {len(legal_files)}")
for p in legal_files:
    print(f"- {p.name}: {p.stat().st_size:,} bytes")

✓ Thư mục đã sẵn sàng: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/landing/legal
Đang tải Luat_Phong_Chong_Ma_Tuy_2021.pdf...
↷ Bỏ qua vì đã tồn tại: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/landing/legal/Luat_Phong_Chong_Ma_Tuy_2021.pdf
Đang tải Luat_120_2025.pdf...
↷ Bỏ qua vì đã tồn tại: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/landing/legal/Luat_120_2025.pdf
Đang tải Nghi_Dinh_28_2026_ND_CP.pdf...
✓ Đã tải: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/landing/legal/Nghi_Dinh_28_2026_ND_CP.pdf (1,471,070 bytes)
Hoàn tất: 3/3 file đã sẵn sàng trong /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/landing/legal
Legal files: 4
- Luat_120_2025.pdf: 10,366,370 bytes
- Luat_Phong_Chong_Ma_Tuy_2021.pdf: 537,045 bytes
- Luat_phong_chong_ma_tuy_Nghi_Duong_Hai_Phong_2025.pdf: 1,026,002 bytes
- Nghi_Dinh_28_2026_ND_CP.pdf: 1,471,070 bytes


## Task 2 - Crawl bài báo

Mục tiêu: crawl tối thiểu 5 bài báo và lưu JSON trong `data/landing/news/`.

In [47]:
import asyncio
from src.task2_crawl_news import DATA_DIR as NEWS_DIR, ARTICLE_URLS, crawl_all

print(f"Configured URLs: {len(ARTICLE_URLS)}")

if RUN_TASK2_CRAWL:
    await crawl_all()

article_jsons = sorted(NEWS_DIR.glob("article_*.json"))
print(f"News JSON files: {len(article_jsons)}")
for p in article_jsons:
    data = json.loads(p.read_text(encoding="utf-8"))
    print(f"- {p.name}: {data.get('title', 'Unknown')} | chars={len(data.get('content_markdown', ''))}")

Configured URLs: 5
[1/5] Crawling: https://thanhnien.vn/ca-si-long-nhat-bi-bat-showbiz-viet-lien-tiep-chan-dong-vi-ma-tuy-18526052013032001.htm


[INIT].... → Crawl4AI 0.8.9 

[FETCH]... ↓ https://thanhnien.vn/ca-si-long-nhat-bi-bat-show...n-tiep-chan-dong-vi-ma-tuy-18526052013032001.htm  | ✓ | ⏱: 1.75s 

[SCRAPE].. ◆ https://thanhnien.vn/ca-si-long-nhat-bi-bat-show...n-tiep-chan-dong-vi-ma-tuy-18526052013032001.htm  | ✓ | ⏱: 0.05s 

[COMPLETE] ● https://thanhnien.vn/ca-si-long-nhat-bi-bat-show...n-tiep-chan-dong-vi-ma-tuy-18526052013032001.htm  | ✓ | ⏱: 1.83s 

...

[INIT].... → Crawl4AI 0.8.9 

[FETCH]... ↓ https://vietnamnet.vn/loat-ca-si-dinh-chat-cam-ma-tuy-pha-huy-nao-bo-nguoi-tre-ra-sao-2518285.html   | ✓ | ⏱: 1.44s 

[SCRAPE].. ◆ https://vietnamnet.vn/loat-ca-si-dinh-chat-cam-ma-tuy-pha-huy-nao-bo-nguoi-tre-ra-sao-2518285.html   | ✓ | ⏱: 0.04s 

[COMPLETE] ● https://vietnamnet.vn/loat-ca-si-dinh-chat-cam-ma-tuy-pha-huy-nao-bo-nguoi-tre-ra-sao-2518285.html   | ✓ | ⏱: 1.49s 

  ✓ Saved: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/landing/news/article_05.json
News JSON files: 5
- article_01.json: Ca sĩ Long Nhật bị bắt, showbiz Việt liên tiếp chấn động vì ma túy | chars=13654
- article_02.json: Khởi tố, bắt tạm giam ca sĩ Long Nhật, Sơn Ngọc Minh vì tổ chức sử dụng ma túy | chars=12206
- article_03.json: Chưa đầy 1 tháng, 3 nghệ sĩ Việt bị khởi tố vì liên quan ma túy gây chấn động | chars=54638
- article_04.json: Nhức nhối loạt nghệ sĩ vướng lao lý vì ma túy: Không chỉ là sa ngã mà còn là sự tổn thương đối với niềm tin công chúng | chars=91752
- article_05.json: Loạt ca sĩ 'dính' chất cấm, ma túy phá hủy não bộ người trẻ ra sao? | chars=26853


## Task 3 - Convert sang Markdown

Mục tiêu: chuyển legal/news sang markdown trong `data/standardized/`.

In [48]:
from src.task3_convert_markdown import OUTPUT_DIR as STANDARDIZED_DIR, convert_all

if RUN_TASK3_CONVERT:
    convert_all()

md_files = sorted(STANDARDIZED_DIR.rglob("*.md"))
print(f"Markdown files: {len(md_files)}")
for p in md_files:
    print(f"- {p.relative_to(STANDARDIZED_DIR)}: {p.stat().st_size:,} bytes")

if md_files:
    print("\nPreview:")
    print(md_files[0].read_text(encoding="utf-8")[:800])

Task 3: Convert to Markdown (MarkItDown)

--- Legal Documents ---
Converting: Luat_Phong_Chong_Ma_Tuy_2021.pdf
  ✓ Saved: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/standardized/legal/Luat_Phong_Chong_Ma_Tuy_2021.md
Converting: Luat_phong_chong_ma_tuy_Nghi_Duong_Hai_Phong_2025.pdf
  ✓ Saved: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/standardized/legal/Luat_phong_chong_ma_tuy_Nghi_Duong_Hai_Phong_2025.md
Converting: Luat_120_2025.pdf
  ✓ Saved: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/standardized/legal/Luat_120_2025.md
Converting: Nghi_Dinh_28_2026_ND_CP.pdf
  ✓ Saved: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/standardized/legal/Nghi_Dinh_28_2026_ND_CP.md

--- News Articles ---
Converting: article_03.json
  ✓ Saved: /Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/data/standardized/news/article_03.md
Converting: article_02.json
  ✓ Saved: /Users

## Task 4 - Load Documents

Mục tiêu: đọc markdown files.

In [49]:
from src.task4_chunking_indexing import load_documents

docs = load_documents()
print(f"Documents: {len(docs)}")

print(docs[0]['content'][:100]) 
print(docs[0]['metadata']) # Print first 100 characters of the content

Documents: 7
4

CÔNG BÁO/Số 567 + 568/Ngày 30-4-2021

QUỐC HỘI

Luật số: 73/2021/QH14

CỘNG HÒA XÃ HỘI CHỦ NGHĨA 
{'source': 'Luat_Phong_Chong_Ma_Tuy_2021.md', 'path': 'legal/Luat_Phong_Chong_Ma_Tuy_2021.md', 'type': 'legal'}


## Task 4 - Chunking

Mục tiêu: chia documents thành chunks.

In [50]:
from src.task4_chunking_indexing import CHUNK_SIZE, CHUNK_OVERLAP, chunk_documents

chunks = chunk_documents(docs)

print(f"Chunks: {len(chunks)}")
print(f"Chunk config: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")

print(chunks[0]['content'][:200])
print(chunks[0]['metadata'])

Chunks: 709
Chunk config: size=500, overlap=50
4

CÔNG BÁO/Số 567 + 568/Ngày 30-4-2021

QUỐC HỘI

Luật số: 73/2021/QH14

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc

LUẬT
PHÒNG, CHỐNG MA TÚY

Căn cứ Hiến pháp nước Cộng hòa xã hộ
{'source': 'Luat_Phong_Chong_Ma_Tuy_2021.md', 'path': 'legal/Luat_Phong_Chong_Ma_Tuy_2021.md', 'type': 'legal', 'chunk_index': 0, 'chunking_method': 'recursive'}


## Task 4 - Embedding và Indexing

Mục tiêu: embed chunks bằng OpenAI và index vào Weaviate Docker.

In [51]:
from src.task4_chunking_indexing import (
    EMBEDDING_MODEL, WEAVIATE_COLLECTION,
    embed_chunks, index_to_vectorstore,
)

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Weaviate collection: {WEAVIATE_COLLECTION}")

if RUN_TASK4_INDEX:
    embedded_chunks = embed_chunks(chunks)
    index_to_vectorstore(embedded_chunks)
    print("Indexed chunks to Weaviate.")
else:
    print("\nSkip indexing. Set RUN_TASK4_INDEX=True to call OpenAI embedding and write Weaviate.")

Embedding model: text-embedding-3-small
Weaviate collection: DrugLawDocs
  Embedded 64/709 chunks
  Embedded 128/709 chunks
  Embedded 192/709 chunks
  Embedded 256/709 chunks
  Embedded 320/709 chunks
  Embedded 384/709 chunks
  Embedded 448/709 chunks
  Embedded 512/709 chunks
  Embedded 576/709 chunks
  Embedded 640/709 chunks
  Embedded 704/709 chunks
  Embedded 709/709 chunks
Indexed chunks to Weaviate.


In [52]:
# Weaviate Docker helper
# Chạy trong terminal nếu Weaviate chưa bật:
#   docker compose up -d weaviate
# Kiểm tra readiness:
import requests

try:
    ready = requests.get("http://localhost:8080/v1/.well-known/ready", timeout=3)
    print("Weaviate ready status:", ready.status_code, ready.text[:200])
except Exception as exc:
    print("Weaviate chưa sẵn sàng:", exc)

Weaviate ready status: 200 


## Task 5 - Semantic Search

Mục tiêu: embed query bằng cùng OpenAI embedding model và tìm trong Weaviate.

In [53]:
from src.task5_semantic_search import semantic_search

query = "hình phạt cho tội tàng trữ ma túy"

if RUN_SEMANTIC_SEARCH:
    semantic_results = semantic_search(query, top_k=3)
    for r in semantic_results:
        print(f"[{r['score']:.3f}] {r['metadata']}\n{r['content'][:300]}\n")
else:
    print("Skip semantic search. Set RUN_SEMANTIC_SEARCH=True after indexing Task 4.")

## Task 6 - Lexical Search BM25

Mục tiêu: tìm kiếm keyword bằng BM25 trên corpus markdown/chunks local.

In [54]:
from src.task6_lexical_search import lexical_search

lexical_results = lexical_search("Điều 248 tàng trữ trái phép chất ma túy", top_k=5)
for r in lexical_results:
    print(f"[{r['score']:.3f}] {r['metadata']}\n{r['content'][:300]}\n")

[20.817] {'source': 'article_03.md', 'path': 'news/article_03.md', 'type': 'news', 'chunk_index': 46, 'chunking_method': 'recursive'}
Ca sĩ Sơn Ngọc Minh
Trong đó, 71 bị can bị khởi tố, bắt tạm giam để điều tra về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng trái phép chất ma túy”; 3 trường hợp còn lại bị xử lý hành chính.

[19.916] {'source': 'article_02.md', 'path': 'news/article_02.md', 'type': 'news', 'chunk_index': 15, 'chunking_method': 'recursive'}
Qua công tác điều tra vụ án “Mua bán trái phép chất ma túy, Tàng trữ trái phép chất ma túy và Tổ chức sử dụng trái phép chất ma túy” phát hiện trên địa bàn TPHCM trong quý I/2026, Phòng Cảnh sát điều tra tội phạm về ma túy Công an TPHCM phát hiện ngoài các đối tượng đã bị xử lý, còn nhiều đối tượng 

[18.000] {'source': 'article_02.md', 'path': 'news/article_02.md', 'type': 'news', 'chunk_index': 18, 'chunking_method': 'recursive'}
Kết quả khám phá bước đầu đã bắt giữ, xử lý 74 đối 

## Task 7 - Reranking bằng Qwen

Mục tiêu: rerank candidates từ retrieval. Nếu chưa cài/tải Qwen thì code dùng fallback keyword để notebook vẫn chạy.

In [55]:
from src.task7_reranking import rerank

reranked = rerank("hình phạt ma túy", lexical_results, top_k=3)
for r in reranked:
    print(f"[{r['score']:.3f}] reranker={r.get('reranker')} | original={r.get('original_score')}\n{r['content'][:300]}\n")

[1.062] reranker=Qwen/Qwen3-Reranker-0.6B | original=18.000069923278282
Kết quả khám phá bước đầu đã bắt giữ, xử lý 74 đối tượng; trong đó đã khởi tố, bắt tạm giam 71 bị can về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng trái phép chất ma túy”; đồng thời phối hợp Công an phường xử lý hành chính 03 đối tượng theo quy 

[-0.438] reranker=Qwen/Qwen3-Reranker-0.6B | original=17.926842216866234
Ngày 20.5, Công an TP.HCM thông tin đã khởi tố, tạm giam 71 bị can trong một đường dây ma túy lớn trên địa bàn [thành phố](https://thanhnien.vn/thanh-pho.html "thành phố"). Trong số này có ca sĩ Long Nhật và ca sĩ Sơn Ngọc Minh - cựu thành viên nhóm V.Music. Các bị can bị điều tra về các hành vi gồm

[-1.188] reranker=Qwen/Qwen3-Reranker-0.6B | original=20.81684128096414
Ca sĩ Sơn Ngọc Minh
Trong đó, 71 bị can bị khởi tố, bắt tạm giam để điều tra về các hành vi “Mua bán trái phép chất ma túy”, “Tàng trữ trái phép chất ma túy” và “Tổ chức sử dụng t

## Task 8 - PageIndex Vectorless Fallback

Mục tiêu: dùng PageIndex nếu có key/SDK; nếu chưa có thì dùng fallback local để xem flow.

In [56]:
from src.task8_pageindex_vectorless import upload_documents, pageindex_search

if RUN_PAGEINDEX_UPLOAD:
    uploaded = upload_documents()
    print(f"Uploaded: {len(uploaded)} documents")

pageindex_results = pageindex_search("ma túy nghệ sĩ", top_k=3)
for r in pageindex_results:
    print(f"[{r['score']:.3f}] source={r['source']} {r['metadata']}\n{r['content'][:300]}\n")

ImportError: cannot import name 'PageIndex' from 'pageindex' (/Users/thangnguyen/Documents/VinAI - Lab/Day08_RAG_pipeline_cohort2/.venv/lib/python3.12/site-packages/pageindex/__init__.py)

## Task 9 - Retrieval Pipeline

Mục tiêu: semantic + lexical -> RRF merge -> rerank -> PageIndex fallback nếu score thấp.

In [ ]:
from src.task9_retrieval_pipeline import retrieve

pipeline_results = retrieve("hình phạt cho tội tàng trữ trái phép chất ma túy", top_k=5)
for r in pipeline_results:
    print(f"[{r['score']:.3f}] source={r['source']} {r.get('metadata', {})}\n{r['content'][:300]}\n")

## Task 10 - Generation có Citation

Mục tiêu: lấy context từ retrieval, reorder để giảm lost-in-the-middle, rồi generate answer có citation. Mặc định cell dưới chỉ preview context; bật `RUN_GENERATION=True` để gọi LLM.

In [ ]:
from src.task10_generation import reorder_for_llm, format_context, generate_with_citation, extractive_answer

question = "Hình phạt cho tội tàng trữ trái phép chất ma túy theo pháp luật Việt Nam?"
context_chunks = reorder_for_llm(pipeline_results)
context = format_context(context_chunks)

print("Reordered context preview:\n")
print(context[:1500])

if RUN_GENERATION:
    result = generate_with_citation(question, top_k=5)
else:
    result = {
        "answer": extractive_answer(question, context_chunks),
        "sources": context_chunks,
        "retrieval_source": context_chunks[0].get("source", "none") if context_chunks else "none",
    }

print("\nAnswer:\n", result["answer"])
print("\nRetrieval source:", result["retrieval_source"])
print("Sources:", len(result["sources"]))

## Quick Checklist

- Task 1/2 tạo raw files trong `data/landing/`.
- Task 3 tạo markdown trong `data/standardized/`.
- Task 4 tạo chunks và, nếu bật flag, index vào Weaviate.
- Task 5/6 trả retrieval results.
- Task 7 rerank candidates.
- Task 8 fallback vectorless.
- Task 9 trả kết quả retrieval hợp nhất.
- Task 10 tạo câu trả lời có citation.